# Usage metering tests

In [1]:
import boto3
import urllib.parse as urlparse 
import json
import shutil
import time

from datetime import datetime, timezone

In [2]:
PROFILE = 'default'
REGION = 'us-east-1'

SESSION = boto3.Session(profile_name=PROFILE, region_name=REGION)

In [3]:
def list_marketplace_products():
    try:
        # Create marketplace catalog client
        mpc = SESSION.client('marketplace-catalog')
        next_token = None
        all_products = []

        paginator = mpc.get_paginator('list_entities')

        pagination_config = {
            'PaginationConfig': {
                'PageSize': 10,   # Number of items per page
            }
        }

        # Define the parameters for the 'list_entities' operation
        operation_parameters = {
            'Catalog': 'AWSMarketplace',  # Required fixed value
            'EntityType': 'SaaSProduct',   # Type of entity to list (e.g., AmiProduct)
            # Add filters or sorting options if needed:
            # 'FilterList': [{'Name': 'EntityId', 'ValueList': ['example-id']}],
            # 'Sort': {'SortBy': 'LastModifiedDate', 'SortOrder': 'DESCENDING'}
        }

        # Paginate through the results
        response_iterator = paginator.paginate(**operation_parameters, **pagination_config)
        #print(response_iterator)
        #print("befor page")
        for page in response_iterator:
            #print(page)
            # Process each page of results
            #print(json.dumps(page.get('EntitySummaryList'), indent=2))
            for entity in page.get('EntitySummaryList'):
                all_products.append(entity)

        return all_products
    except Exception as error:
        print("Error listing marketplace products:", error)
        return []

def get_product_for_product_code(product_code):
    try:
        products = list_marketplace_products()
        mpc = SESSION.client('marketplace-catalog')

        for product in products:
            #print(product)

            product_data = mpc.describe_entity(
                Catalog='AWSMarketplace',
                EntityId=product.get('EntityId')
            )

            print("EntityId:", product.get('EntityId'))
            print("ProductCode:", product_data.get('DetailsDocument').get('Description').get('ProductCode'))
            
            if product_data.get('DetailsDocument').get('Description').get('ProductCode') == product_code:
                print("FOUND Product for code:", product_code, "productId:", product.get('EntityId'))
                return product_data
            
            time.sleep(0.1)
        
        return {}
    except Exception as error:
        print("Error getting id for product code:", error)
        return {}

In [4]:
product_code = 'cqj79vcg9mig83heufcyjpfai' # My SaaS Product - Contract with Consumption - Landing Page Test 2

#product_code = pp_product_code
#product_code = 'bcmatrsompluro3g7diajhey9'
mpe = SESSION.client('marketplace-entitlement')
response = mpe.get_entitlements(
    ProductCode=product_code,
        
)
print(json.dumps(response,indent=2, default=str))

{
  "Entitlements": [
    {
      "ProductCode": "cqj79vcg9mig83heufcyjpfai",
      "Dimension": "dimension_1_id",
      "CustomerIdentifier": "767ELa4tizH",
      "CustomerAWSAccountId": "305142167625",
      "Value": {
        "IntegerValue": 1
      },
      "ExpirationDate": "2025-09-11 16:27:02.714000+00:00"
    }
  ],
  "ResponseMetadata": {
    "RequestId": "5fc5967a-9db3-4e79-8935-474425232ebd",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "x-amzn-requestid": "5fc5967a-9db3-4e79-8935-474425232ebd",
      "content-type": "application/x-amz-json-1.1",
      "content-length": "254",
      "date": "Mon, 11 Aug 2025 16:30:04 GMT"
    },
    "RetryAttempts": 0
  }
}


## Metering into DDB

Put metering records in a DDB table. Replace `table_name` and the value for `dimension` with your product defintion.

In [5]:
def create_metering_item(customer_aws_account_id, dimension_name):
    item = {
        "create_timestamp": {
            "N": f"{int(time.time())}"
        },
        "customerIdentifier": {
            "S": customer_aws_account_id
        },
        "dimension_usage": {
            "L": [
            {
                "M": {
                "dimension": {
                    "S": dimension_name
                },
                "value": {
                    "N": "3"
                }
                }
            }
            ]
        },
        "metering_pending": {
            "S": "true"
        }
    }
    
    return item


In [7]:
#table_name = 'AWSMarketplaceMeteringRecords'
table_name = 'AWSMarketplaceMeteringRecordsIdentifier'
ddb = SESSION.client('dynamodb')

In [8]:
item = create_metering_item('305142167625', 'metered_1_id')
print(json.dumps(item, indent=2, default=str))

response = ddb.put_item(
    TableName=table_name,
    Item=item
)
print(json.dumps(response, indent=2, default=str))

{
  "create_timestamp": {
    "N": "1754930047"
  },
  "customerIdentifier": {
    "S": "305142167625"
  },
  "dimension_usage": {
    "L": [
      {
        "M": {
          "dimension": {
            "S": "metered_1_id"
          },
          "value": {
            "N": "3"
          }
        }
      }
    ]
  },
  "metering_pending": {
    "S": "true"
  }
}
{
  "ResponseMetadata": {
    "RequestId": "PAKNUTA14PEESUGVN2JU5FK447VV4KQNSO5AEMVJF66Q9ASUAAJG",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "server": "Server",
      "date": "Mon, 11 Aug 2025 16:34:07 GMT",
      "content-type": "application/x-amz-json-1.0",
      "content-length": "2",
      "connection": "keep-alive",
      "x-amzn-requestid": "PAKNUTA14PEESUGVN2JU5FK447VV4KQNSO5AEMVJF66Q9ASUAAJG",
      "x-amz-crc32": "2745614147"
    },
    "RetryAttempts": 0
  }
}


In [13]:
stack_name = 'saas-integration-update-identifier'
cf_client = boto3.client('cloudformation')
paginator = cf_client.get_paginator('list_stack_resources')

for page in paginator.paginate(StackName=stack_name):
    for resource in page['StackResourceSummaries']:
        print(json.dumps(resource, indent=2, default=str))
        print(f"{resource['ResourceType']}: {resource['LogicalResourceId']}")

{
  "LogicalResourceId": "AWSMarketplaceMeteringRecords",
  "PhysicalResourceId": "AWSMarketplaceMeteringRecordsIdentifier",
  "ResourceType": "AWS::DynamoDB::Table",
  "LastUpdatedTimestamp": "2025-08-11 15:48:46.864000+00:00",
  "ResourceStatus": "CREATE_COMPLETE",
  "DriftInformation": {
    "StackResourceDriftStatus": "NOT_CHECKED"
  }
}
AWS::DynamoDB::Table: AWSMarketplaceMeteringRecords
{
  "LogicalResourceId": "AWSMarketplaceSubscribers",
  "PhysicalResourceId": "AWSMarketplaceSubscribersIdentifier",
  "ResourceType": "AWS::DynamoDB::Table",
  "LastUpdatedTimestamp": "2025-08-11 15:48:26.517000+00:00",
  "ResourceStatus": "CREATE_COMPLETE",
  "DriftInformation": {
    "StackResourceDriftStatus": "NOT_CHECKED"
  }
}
AWS::DynamoDB::Table: AWSMarketplaceSubscribers
{
  "LogicalResourceId": "CAPILambdasExecutionRole",
  "PhysicalResourceId": "saas-integration-update-id-CAPILambdasExecutionRole-Kg2RlbbJwDpQ",
  "ResourceType": "AWS::IAM::Role",
  "LastUpdatedTimestamp": "2025-08-11 1